# C-MAPSS FD003

**Dataset:** Single operating condition · Two fault modes (HPC + Fan degradation)  
**Method:** ORTHON feature matrix → 3-model stacking ensemble (LightGBM + XGBoost + HistGB → RidgeCV)  
**Reproducibility:** Seeds 0–29, no selection bias. Random seed threaded through full ML ensemble.  
**IP note:** Feature engineering pipeline not included. Anonymised F-columns only. Patent Pending.

---

In [1]:
# ── Configure CMAPSS data location ────────────────────────────────────────
# Enter the path to the directory that contains the fd001/, fd002/, fd003/,
# and fd004/ subdirectories on your machine.
#
# Examples:
#   ~/domains/cmapss        (if you keep datasets under ~/domains/)
#   ~/cmapss/data           (if you cloned this repo and use the bundled data/)
#
# Press Enter to accept the default.

from pathlib import Path

DEFAULT_ROOT = "~/cmapss/data"
_user_input = (lambda *_: "~/cmapss/data")(f"CMAPSS data root [{DEFAULT_ROOT}]: ").strip() or DEFAULT_ROOT
CMAPSS_ROOT = Path(_user_input).expanduser()

assert CMAPSS_ROOT.exists(), f"Directory not found: {CMAPSS_ROOT}"
print(f"Using CMAPSS root: {CMAPSS_ROOT}")


Using CMAPSS root: /Users/jasonrudder/cmapss/data


In [2]:
# ── Choose which seeds to run ─────────────────────────────────────────────
# Enter the seed(s) you want to run.
#
# Examples:
#   0          ← single seed (fastest, ~1 minute)
#   5          ← any single seed
#   0-4        ← five seeds
#   0-29       ← full 30-seed published sweep (~25 minutes for FD002/FD004)
#
# Press Enter to accept the default.

DEFAULT_SEEDS = "0-29"
_seed_input = (lambda *_: "0-29")(f"Seeds to run [{DEFAULT_SEEDS}]: ").strip() or DEFAULT_SEEDS

if "-" in _seed_input:
    _start, _end = _seed_input.split("-", 1)
    SEEDS = list(range(int(_start), int(_end) + 1))
else:
    SEEDS = [int(_seed_input)]

assert len(SEEDS) > 0, "No seeds selected."
print(f"Will run {len(SEEDS)} seed(s): {SEEDS}")
if len(SEEDS) == 1:
    print("Note: single-seed mode — the multi-seed plots and stats further down")
    print("      will degenerate (std=0, single bar). The numeric result is still correct.")


Will run 30 seed(s): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from pathlib import Path

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "monospace",
})

print("All imports OK.")

All imports OK.


## 1 · Load Data

In [4]:
DATA_DIR = CMAPSS_ROOT / "fd003"

train_df = pl.read_parquet(DATA_DIR / "train.parquet")
test_df  = pl.read_parquet(DATA_DIR / "test.parquet")

feature_cols = [c for c in train_df.columns if c.startswith("F")]
print(f"Train rows : {len(train_df):,}")
print(f"Test rows  : {len(test_df):,}")
print(f"Features   : {len(feature_cols)}")
print(f"Engines    : {train_df['cohort'].n_unique()} train  /  {test_df['cohort'].n_unique()} test")
train_df.head(3)

Train rows : 24,720
Test rows  : 100
Features   : 149
Engines    : 100 train  /  100 test


cohort,F001,F002,F003,F004,F005,F006,F007,F008,F009,F010,F011,F012,F013,F014,F015,F016,F017,F018,F019,F020,F021,F022,F023,F024,F025,F026,F027,F028,F029,F030,F031,F032,F033,F034,F035,F036,…,F114,F115,F116,F117,F118,F119,F120,F121,F122,F123,F124,F125,F126,F127,F128,F129,F130,F131,F132,F133,F134,F135,F136,F137,F138,F139,F140,F141,F142,F143,F144,F145,F146,F147,F148,F149,RUL
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""engine_1""",0.009132,-0.192601,-0.152215,0.008337,-0.137183,0.014996,-0.222303,-0.079574,-0.005635,0.002453,-0.200666,-0.010864,-0.10752,-0.181001,-0.107112,-0.022744,0.02023,-0.190654,-0.195894,-0.123472,-0.044531,0.019857,-0.157151,-0.106858,-0.178971,-0.210127,-0.016481,-0.085985,-0.591226,-0.591226,-0.01623,-0.193182,-0.1702,-0.335409,-0.206434,-0.011077,…,-0.124175,-0.049251,-0.035445,0.693719,-0.029753,-0.029753,-0.00033,-0.000309,-0.000315,-0.000325,0.010037,0.010037,-0.0106,0.003796,-0.000422,-0.000728,-0.000362,-0.001272,-0.000131,-0.001051,0.000075,-0.001834,-0.000225,-0.004633,0.004994,0.002202,0.008887,0.016287,-0.003518,-0.00011,0.009331,-0.000254,0.009065,-0.01037,-0.000376,0.000403,125.0
"""engine_1""",0.009132,-0.192601,-0.152215,0.008337,-0.137183,0.014996,-0.222303,-0.079574,-0.005635,0.002453,-0.200666,-0.010864,-0.10752,-0.181001,-0.107112,-0.022744,0.02023,-0.190654,-0.195894,-0.123472,-0.044531,0.019857,-0.157151,-0.106858,-0.178971,-0.210127,-0.016481,-0.085985,-0.667816,-0.667816,-0.01623,-0.193182,-0.1702,-0.335409,-0.206434,-0.011077,…,-0.124175,-0.049251,-0.035445,0.693719,-0.674787,-0.674787,-0.00033,-0.000309,-0.000315,-0.000325,0.010037,0.010037,-0.408256,0.003796,-0.000422,-0.000728,-0.000362,-0.001272,-0.000131,-0.871373,0.000075,-0.001834,-0.000225,-0.004633,0.004994,-0.461163,0.008887,0.016287,-0.003518,-0.00011,0.009331,-0.000254,0.009065,-0.01037,-0.000376,0.000403,125.0
"""engine_1""",0.009132,-0.192601,-0.152215,0.008337,-0.137183,0.014996,-0.222303,-0.079574,-0.005635,0.002453,-0.200666,-0.010864,-0.10752,-0.181001,-0.107112,-0.022744,0.02023,-0.190654,-0.195894,-0.123472,-0.044531,0.019857,-0.157151,-0.106858,-0.178971,-0.210127,-0.016481,-0.085985,-0.674347,-0.674347,-0.01623,-0.193182,-0.1702,-0.335409,-0.206434,-0.011077,…,-0.124175,-0.049251,-0.035445,0.693719,-0.152449,-0.152449,-0.00033,-0.321419,-0.000315,-0.065049,0.303725,0.303725,-0.622061,0.003796,-0.869233,-0.552902,-0.000362,-0.001272,-0.000131,-0.348573,-0.080026,-0.001834,-0.6279,-0.263066,0.004994,-0.91437,0.865349,-0.126805,-0.003518,0.303801,0.009331,-0.624181,0.869037,-0.01037,-0.000376,-0.384713,125.0


## 2 · ML Pipeline — Seeds 0–29

In [5]:
def nasa_score(y_true, y_pred):
    """NASA asymmetric scoring function. Penalises late predictions more."""
    d = y_pred - y_true
    scores = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return float(np.sum(scores))


RUL_CAP = 125.0
N_SPLITS = 5


def run_seed(seed, X_train, y_train, groups, X_test, y_test):
    # Mirrors run.py's get_base_models + stacking logic exactly so the
    # notebook and the CLI report identical numbers for the same seed.
    # n_jobs=1 + deterministic=True pin thread-count-independent bit-identity.
    lgbm = LGBMRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        num_leaves=31, min_child_samples=10, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        verbose=-1, random_state=seed,
        n_jobs=1, deterministic=True,
    )
    xgb  = XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        min_child_weight=10, verbosity=0, random_state=seed,
        n_jobs=1,
    )
    hgb  = HistGradientBoostingRegressor(
        max_iter=500, max_depth=6, learning_rate=0.05,
        min_samples_leaf=10, random_state=seed,
    )
    base_models = {"hist": hgb, "lgb": lgbm, "xgb": xgb}
    names = list(base_models.keys())

    # Clip training targets at RUL_CAP (matches run.py)
    y_train_clipped = np.clip(y_train.astype(np.float64), 0, RUL_CAP)

    gkf = GroupKFold(n_splits=N_SPLITS)
    n_train, n_test = len(X_train), len(X_test)
    oof        = {n: np.zeros(n_train) for n in names}
    test_preds = {n: np.zeros(n_test)  for n in names}

    for tr_idx, va_idx in gkf.split(X_train, y_train_clipped, groups):
        Xtr, Xva = X_train[tr_idx], X_train[va_idx]
        ytr      = y_train_clipped[tr_idx]
        for name, model in base_models.items():
            clone = type(model)(**model.get_params())
            clone.fit(Xtr, ytr)
            oof[name][va_idx] = clone.predict(Xva)
            test_preds[name] += clone.predict(X_test) / N_SPLITS

    oof_stack  = np.column_stack([oof[n]        for n in names])
    test_stack = np.column_stack([test_preds[n] for n in names])
    ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0])
    ridge.fit(oof_stack, y_train_clipped)
    preds = np.clip(ridge.predict(test_stack), 0, RUL_CAP)

    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    nasa = nasa_score(y_test, preds)
    return rmse, nasa, preds


# ── Prepare arrays ────────────────────────────────────────────────────────
X_train = train_df.select(feature_cols).to_numpy()
y_train = train_df["RUL"].to_numpy()
groups  = train_df["cohort"].to_numpy()
X_test  = test_df.select(feature_cols).to_numpy()
y_test  = test_df["RUL"].to_numpy()

results = []

print(f"Running {len(SEEDS)} seeds ...")
for s in SEEDS:
    rmse, nasa, preds = run_seed(s, X_train, y_train, groups, X_test, y_test)
    results.append({"seed": s, "rmse": rmse, "nasa": nasa, "preds": preds})
    if s % 5 == 4:
        print(f"  seeds 0–{s} done")

rmse_vals = np.array([r["rmse"] for r in results])
nasa_vals = np.array([r["nasa"] for r in results])
best_idx  = int(np.argmin(rmse_vals))
best_pred = results[best_idx]["preds"]

print(f"\nRMSE  mean={rmse_vals.mean():.2f}  std={rmse_vals.std():.2f}  "
      f"min={rmse_vals.min():.2f}  max={rmse_vals.max():.2f}")
print(f"NASA  mean={nasa_vals.mean():.0f}  "
      f"min={nasa_vals.min():.0f}  max={nasa_vals.max():.0f}")
print(f"Zero catastrophic failures: {(rmse_vals > 50).sum() == 0}")

Running 30 seeds ...


  File "/Users/jasonrudder/cmapss/.venv/lib/python3.14/site-packages/joblib/externals/loky/backend/context.py", line 249, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
  File "/Users/jasonrudder/cmapss/.venv/lib/python3.14/site-packages/joblib/externals/loky/backend/context.py", line 312, in _count_physical_cores_darwin
    cpu_info = subprocess.run(
        "sysctl -n hw.physicalcpu".split(),
        capture_output=True,
        text=True,
    )
  File "/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14/subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14/subprocess.py", line 1038, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                

  seeds 0–4 done


  seeds 0–9 done


  seeds 0–14 done


  seeds 0–19 done


  seeds 0–24 done


  seeds 0–29 done

RMSE  mean=10.69  std=0.05  min=10.54  max=10.79
NASA  mean=184  min=177  max=188
Zero catastrophic failures: True


## 3 · Summary Statistics

In [6]:
import pandas as pd

summary = pd.DataFrame({
    "Metric": ["RMSE mean", "RMSE std", "RMSE min", "RMSE max",
               "NASA mean", "NASA min", "NASA max",
               "Seeds run", "Catastrophic failures (RMSE>50)"],
    "Value": [
        f"{rmse_vals.mean():.3f}",
        f"{rmse_vals.std():.3f}",
        f"{rmse_vals.min():.3f}",
        f"{rmse_vals.max():.3f}",
        f"{nasa_vals.mean():.1f}",
        f"{nasa_vals.min():.1f}",
        f"{nasa_vals.max():.1f}",
        "30 (seeds 0–29)",
        str(int((rmse_vals > 50).sum())),
    ]
})
summary.style.set_properties(**{"text-align": "left"})

,Metric,Value
0,RMSE mean,10.688
1,RMSE std,0.052
2,RMSE min,10.543
3,RMSE max,10.793
4,NASA mean,184.4
5,NASA min,176.6
6,NASA max,188.3
7,Seeds run,30 (seeds 0–29)
8,Catastrophic failures (RMSE>50),0


## 4 · Plots

In [ ]:
fig = plt.figure(figsize=(15, 11))
fig.suptitle("C-MAPSS FD003 — ORTHON + Stacking Ensemble", fontsize=14, y=1.01)
gs = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.35)

# ── (A) RMSE across all 30 seeds ─────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
colors = ["#27ae60" if v == rmse_vals.min() else "#3498db" for v in rmse_vals]
ax1.bar(SEEDS, rmse_vals, color=colors, edgecolor="none", width=0.7)
ax1.axhline(rmse_vals.mean(), color="#2c3e50", linewidth=1.5,
            linestyle="--", label=f"Mean {rmse_vals.mean():.2f}")
ax1.set_title("RMSE — All 30 Seeds", fontsize=11, pad=8)
ax1.set_xlabel("Seed"); ax1.set_ylabel("RMSE")
ax1.legend(fontsize=9)
_rmse_top = rmse_vals.max() + (rmse_vals.max() - rmse_vals.min()) * 0.18
ax1.set_ylim(top=rmse_vals.max() + (rmse_vals.max() - rmse_vals.min()) * 0.30)
ax1.annotate(f"Best: {rmse_vals.min():.2f} (seed {best_idx})",
             xy=(best_idx, rmse_vals.min()),
             xytext=(best_idx + 2, _rmse_top),
             fontsize=8, arrowprops=dict(arrowstyle="->", color="#27ae60"),
             color="#27ae60")

# ── (B) Predicted vs Actual RUL (best seed) ───────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(y_test, best_pred, alpha=0.35, s=8, color="#3498db", label="Engine")
lim = max(y_test.max(), best_pred.max()) * 1.05
ax2.plot([0, lim], [0, lim], "k--", linewidth=1, label="Perfect")
ax2.set_title(f"Predicted vs Actual RUL (seed {best_idx})", fontsize=11, pad=8)
ax2.set_xlabel("Actual RUL"); ax2.set_ylabel("Predicted RUL")
ax2.legend(fontsize=9)

# ── (C) Error histogram ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
errors = best_pred - y_test
ax3.hist(errors, bins=40, color="#2ecc71", edgecolor="white", linewidth=0.4)
ax3.axvline(0, color="#e74c3c", linewidth=1.5, label="Zero error")
ax3.axvline(errors.mean(), color="#e67e22", linewidth=1.5,
            linestyle="--", label=f"Mean {errors.mean():.2f}")
ax3.set_title("Residual Distribution (pred − actual)", fontsize=11, pad=8)
ax3.set_xlabel("Error (cycles)"); ax3.set_ylabel("Count")
ax3.legend(fontsize=9)

# ── (D) RMSE distribution (histogram across seeds) ───────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.hist(rmse_vals, bins=15, color="#9b59b6", edgecolor="white", linewidth=0.4)
ax4.axvline(rmse_vals.mean(), color="#2c3e50", linewidth=1.5,
            linestyle="--", label=f"Mean {rmse_vals.mean():.2f}")
ax4.set_title("RMSE Distribution Across Seeds", fontsize=11, pad=8)
ax4.set_xlabel("RMSE"); ax4.set_ylabel("Frequency")
ax4.legend(fontsize=9)

plt.savefig("fd003_results.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fd003_results.png")

## NASA Score — FD003


In [ ]:
# ── NASA score visualisations ─────────────────────────────────────────────
# NASA PHM08 asymmetric scoring: late predictions cost much more than early ones.
#   late  (pred > actual):  exp(error / 10) − 1
#   early (pred < actual):  exp(−error / 13) − 1
# Lower NASA = better. Total NASA = sum of per-engine penalties.

fig = plt.figure(figsize=(15, 11))
fig.suptitle("C-MAPSS FD003 — NASA Score Analysis", fontsize=14, y=1.01)
gs = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.35)

best_nasa_idx = int(np.argmin(nasa_vals))

# ── (A) NASA score per seed ──────────────────────────────────────────────
axN1 = fig.add_subplot(gs[0, 0])
nasa_colors = ["#27ae60" if v == nasa_vals.min() else "#9b59b6" for v in nasa_vals]
axN1.bar(SEEDS, nasa_vals, color=nasa_colors, edgecolor="none", width=0.7)
axN1.axhline(nasa_vals.mean(), color="#2c3e50", linewidth=1.5,
             linestyle="--", label=f"Mean {nasa_vals.mean():.0f}")
axN1.set_title(f"NASA Score — {len(SEEDS)} Seed(s)", fontsize=11, pad=8)
axN1.set_xlabel("Seed"); axN1.set_ylabel("NASA score (lower = better)")
axN1.legend(fontsize=9)
if len(SEEDS) > 1:
    _nasa_top = nasa_vals.max() + np.ptp(nasa_vals) * 0.18
    axN1.set_ylim(top=nasa_vals.max() + np.ptp(nasa_vals) * 0.30)
    axN1.annotate(f"Best: {nasa_vals.min():.0f} (seed {SEEDS[best_nasa_idx]})",
                  xy=(SEEDS[best_nasa_idx], nasa_vals.min()),
                  xytext=(SEEDS[best_nasa_idx] + max(1, len(SEEDS)//15),
                          _nasa_top),
                  fontsize=8, arrowprops=dict(arrowstyle="->", color="#27ae60"),
                  color="#27ae60")

# ── (B) NASA distribution across seeds ────────────────────────────────────
axN2 = fig.add_subplot(gs[0, 1])
if len(SEEDS) > 1:
    axN2.hist(nasa_vals, bins=min(15, len(SEEDS)), color="#9b59b6",
              edgecolor="white", linewidth=0.4)
    axN2.axvline(nasa_vals.mean(), color="#2c3e50", linewidth=1.5,
                 linestyle="--", label=f"Mean {nasa_vals.mean():.0f}")
    axN2.legend(fontsize=9)
else:
    axN2.text(0.5, 0.5, f"NASA = {nasa_vals[0]:.0f}\n(single seed)",
              ha="center", va="center", transform=axN2.transAxes, fontsize=12)
axN2.set_title("NASA Distribution Across Seeds", fontsize=11, pad=8)
axN2.set_xlabel("NASA score"); axN2.set_ylabel("Frequency")

# ── (C) NASA penalty function ─────────────────────────────────────────────
axN3 = fig.add_subplot(gs[1, 0])
err_range = np.linspace(-30, 30, 400)
nasa_curve = np.where(err_range < 0,
                      np.exp(-err_range / 13) - 1,
                      np.exp(err_range / 10) - 1)
axN3.plot(err_range[err_range < 0], nasa_curve[err_range < 0],
          color="#27ae60", linewidth=2, label="Early (pred < actual)")
axN3.plot(err_range[err_range >= 0], nasa_curve[err_range >= 0],
          color="#e74c3c", linewidth=2, label="Late (pred > actual)")
axN3.axvline(0, color="#2c3e50", linewidth=0.8)
axN3.set_title("NASA Penalty Function — Asymmetric", fontsize=11, pad=8)
axN3.set_xlabel("Prediction error (cycles)")
axN3.set_ylabel("Per-engine penalty")
axN3.legend(fontsize=9)

# ── (D) Top per-engine NASA contributors (best seed) ──────────────────────
axN4 = fig.add_subplot(gs[1, 1])
nasa_best_pred = results[best_nasa_idx]["preds"]
per_engine_err = nasa_best_pred - y_test
per_engine_nasa = np.where(per_engine_err < 0,
                           np.exp(-per_engine_err / 13) - 1,
                           np.exp(per_engine_err / 10) - 1)
order = np.argsort(per_engine_nasa)[::-1]
top_n = min(20, len(per_engine_nasa))
top_idx = order[:top_n]
bar_colors = ["#e74c3c" if per_engine_err[i] > 0 else "#27ae60" for i in top_idx]
axN4.bar(range(top_n), per_engine_nasa[top_idx], color=bar_colors, edgecolor="none")
axN4.set_title(f"Top {top_n} Engine Penalties (best NASA seed {SEEDS[best_nasa_idx]})",
               fontsize=11, pad=8)
axN4.set_xlabel("Engine rank (largest penalty first)")
axN4.set_ylabel("Per-engine NASA penalty")
red_patch = plt.Rectangle((0,0),1,1, color="#e74c3c", label="Late (worse)")
green_patch = plt.Rectangle((0,0),1,1, color="#27ae60", label="Early")
axN4.legend(handles=[red_patch, green_patch], fontsize=9)

plt.savefig("fd003_nasa.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved: fd003_nasa.png")
print(f"Total NASA (best seed {SEEDS[best_nasa_idx]}): {nasa_vals[best_nasa_idx]:.1f}")


---
## Reproducibility Statement

All results above are bit-identical across runs on the same hardware.  
Seeds 0–29 were run without selection — this is the full distribution, not a best-of-N report.  
The feature engineering pipeline that produced the anonymised `F001..F{NNN}` columns is not  
included in this repository (Patent Pending). The ML code above is the complete reproduction path.

**Hardware:** Apple M4 Mac Mini 16 GB  
**Dataset:** NASA C-MAPSS FD003 (Single operating condition · Two fault modes (HPC + Fan degradation))  
**License:** CC BY-NC 4.0 + Patent Pending
